# Embeddingi perfum – WSTĘP (badania i przygotowanie)

Ten notatnik to **część badawcza**. Sprawdza dane, buduje cechy i na pomiarach uzasadnia każdą wartość, która potem trafia do `CONFIG` w notatniku **`embeddings_v3_proste`**. Same osadzenia (wynik końcowy, sąsiedzi, zapis plików) powstają dopiero tam – tutaj ich nie ma.

**Układ:**

| Część | Pytanie, na które odpowiada | Co przechodzi do v3 |
|---|---|---|
| 1–3. Dane i cechy | czy dane są poprawne, jak zamieniamy perfumy na liczby | sposób budowy `X_note`, `X_acc` |
| 4. Funkcje modeli | jak działają metody A, B, B\* i hybryda | funkcje `build_A`, `build_B`, `hybrid` (w v3 uproszczone) |
| 5. Ocena | jak uczciwie mierzyć jakość, skoro nie ma etykiet podobieństwa | funkcje `knn_top`, `ocena` |
| 6. Ablacje | które ustawienia mają znaczenie, a które nie | wartości w `CONFIG` |
| 7. Wybór udziału A w hybrydzie | ile metody A w wyniku końcowym | `HYBRID_W_A` |
| 8. Podsumowanie | tabela decyzji | – |

**Trzy miary, tylko tyle potrzeba do decyzji:**

| Miara | Co mówi | Kierunek |
|---|---|---|
| **Zgodność akordów** | czy sąsiedzi perfum mają podobne akordy (poziom losowy ≈ 0,23) | wyżej = lepiej |
| **Bracia w top-10** | jaka część wariantów tej samej linii (ta sama marka i początek nazwy) jest w 10 najbliższych | wyżej = lepiej |
| **Skośność hubness** | czy niektóre perfumy nie „zaśmiecają” cudzych list sąsiadów | niżej = lepiej |

**Wagi warstw piramidy 1/1/1** są ustalone wcześniej – ablacja w części 6 tylko to dokumentuje. **Scalania nut szczegółowych w ogólne tu nie robimy** (zrobisz je przy budowie bazy).

## 0. Importy i ustawienia

`CONFIG` zbiera wszystkie parametry w jednym miejscu; reszta kodu tylko je czyta. Wartości po prawej to te, które część 6 i 7 potwierdza (lub koryguje). Dwa parametry są nowe względem poprzedniej wersji: `B_TOP1_WEIGHT` i `B_PPMI_SHIFT` (ulepszenie B\* z części 6.4), oraz `HYBRID_W_A` (część 7). Komentarz przy `B_ACCORD_GAIN` poprawiono: to mnożnik wagi akordów przy uśrednianiu, a nie „bonus rodziny zapachowej”.

In [1]:
import json
import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import skew

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

CONFIG = dict(
    INPUT_CSV_CANDIDATES = ["../dane/fra_cleaned_final.csv", "fra_cleaned_final.csv", "dane/fra_cleaned_final.csv"],

    LAYER_WEIGHTS = {"Top": 1.0, "Middle": 1.0, "Base": 1.0}, # Równa waga dla każdej warstwy piramidy (ustalone)
    LENGTH_NORM_EXP = 0.0,                                    # Brak kary za długość listy składników
    MIN_DF_NOTE = 3,                                          # Nuta musi wystąpić min. 3 razy w całej bazie
    ACCORD_RANK_WEIGHTS = [1.0, 0.75, 0.56, 0.42, 0.32],      # Malejące wagi dla kolejnych głównych akordów, 0.75^k
    IDF_POWER = 0.5,                                          # Pierwiastek wygładzający wagę rzadkich nut, osłabia nuty dominujące np. piżmo
    A_ALPHA_ACCORD = 0.4,                                     # Metoda A: 40% wpływu akordów, 60% nut
    B_CTX_SMOOTH = 0.5,                                       # Metoda B: Wygładzanie rozkładu kontekstów w PPMI
    B_ACCORD_GAIN = 1.5,                                      # Metoda B: mnożnik wagi akordów przy uśrednianiu wektorów tokenów
    B_TOP1_WEIGHT = 0.75,                                     # Metoda B*: waga dodatkowego tokena "akord nr 1"
    B_PPMI_SHIFT = 0.69,                                      # Metoda B*: przesunięcie PMI przed obcięciem do zera (ok. ln 2)
    HYBRID_W_A = 0.30,                                        # Hybryda: udział metody A (reszta to B*)
    DIM = 256,                                                # Docelowa liczba wymiarów wektora zapachu (kompresja SVD)
    SEED = 42,                                                # Stałe ziarno losowości dla powtarzalności wyników
    N_SUBSAMPLE = 8000,                                       # Liczba zapytań w ablacjach (szybkość)
)
INPUT_CSV = next((p for p in CONFIG["INPUT_CSV_CANDIDATES"] if os.path.exists(p)), None)
assert INPUT_CSV is not None, "Nie znaleziono fra_cleaned_final.csv - dopisz ścieżkę w CONFIG"
np.random.seed(CONFIG["SEED"])
print("plik wejściowy:", INPUT_CSV)

plik wejściowy: fra_cleaned_final.csv

## 1. Wczytanie i audyt danych

**Cel:** sprawdzić, czy tabela jest taka, jak zakładamy, i zobaczyć z góry problemy, które psują ocenę.
* Zamiast sztywnego numeru wiersza (jak w starej wersji) losujemy jeden przykład ziarnem z `CONFIG` – kod nie zależy od kolejności wierszy.
* **Duplikaty `(Perfume, Brand)`** to zwykle wersje męska i damska tego samego zapachu; zostają w danych (decyzja projektowa), ale mają prawie identyczne wektory i w listach sąsiadów zajmują miejsca.
* **„Bracia”** (flankery) – perfumy tej samej marki o wspólnym początku nazwy; będą potrzebne w mierze „bracia w top-10”.

In [2]:
df = pd.read_csv(INPUT_CSV, sep=";", encoding="utf-8-sig")
N = len(df)

print("kształt tabeli (wiersze, kolumny):", df.shape)
print("\nnazwy kolumn:")
print(df.columns.tolist())

print("\nlosowy przykład (ziarno z CONFIG):")
print(df.sample(1, random_state=CONFIG["SEED"]).T)

braki = df.isna().sum()
print("\nbraki danych:")
print(braki[braki > 0])

print("\npowtórzone adresy url:", df.url.duplicated().sum())
print("powtórzone pary (Perfume, Brand):", df.duplicated(["Perfume", "Brand"]).sum())
klucz = df.Brand + "|" + df.Perfume.str.split().str[:2].str.join(" ")
print("perfumy w grupach 'braci' (ta sama marka + dwa pierwsze słowa nazwy):", int((klucz.map(klucz.value_counts()) > 1).sum()))

kształt tabeli (wiersze, kolumny): (24046, 17)

nazwy kolumn:
['url', 'Perfume', 'Brand', 'Country', 'Gender', 'Rating Value', 'Rating Count', 'Year', 'Top', 'Middle', 'Base', 'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5', 'Perfumers']

losowy przykład (ziarno z CONFIG):
                                                          23396
url           https://www.fragrantica.com/perfume/roja-dove/...
Perfume                                 KINGDOM OF SAUDI ARABIA
Brand                                                 ROJA DOVE
Country                                                      UK
Gender                                                   unisex
Rating Value                                               4.47
Rating Count                                                160
Year                                                     2017.0
Top                 mandarin orange, artemisia, lime, aldehydes
Middle        strawberry, banana, raspberry, plum, apple, bl


**Wniosek:** brak powtórzonych adresów url; 217 powtórzonych par `(Perfume, Brand)` (głównie wersje męska i damska) i 7685 perfum w grupach „braci”. Braki są tylko w `Year` (2034) i w `mainaccord2..5`.


## 2. Parsowanie nut i akordów

**Cel:** zamienić teksty `"saffron, cinnamon, nutmeg"` na listy. Nuty: małe litery, bez „unknown”. Akordy: lista do 5 pozycji **w kolejności ważności** (kolejność z kolumn `mainaccord1..5` zostaje zachowana – to informacja, którą wykorzystują wagi rangowe).

Brak `NOTE_MAP` w tej wersji: nuty zostają szczegółowe. Statystyki pokazują, ile nut jest rzadkich (to uzasadnia `MIN_DF_NOTE`).

In [3]:
def split_list(s):
    if not isinstance(s, str):
        return []
    return [x.strip().lower() for x in s.split(",")
            if x.strip() and x.strip().lower() != "unknown"]

for layer in ["Top", "Middle", "Base"]:
    df[layer + "_l"] = df[layer].map(split_list)

acc_cols = [f"mainaccord{i}" for i in range(1, 6)]
df["accords_l"] = df[acc_cols].apply(
    lambda row: [str(x).strip().lower() for x in row if isinstance(x, str) and x.strip()],
    axis=1,
)

all_notes = Counter(n for layer in ["Top_l", "Middle_l", "Base_l"]
                    for lst in df[layer] for n in lst)
all_acc = Counter(a for lst in df["accords_l"] for a in lst)
counts = np.array(list(all_notes.values()))

print(f"unikalne nuty: {len(all_notes)}")
print(f"  występujące tylko raz: {(counts == 1).sum()}")
print(f"  występujące co najmniej 3 razy: {(counts >= 3).sum()}")
print(f"  występujące co najmniej 50 razy: {(counts >= 50).sum()}")
print(f"unikalne akordy: {len(all_acc)}")

print("\nśrednia liczba nut w warstwie:",
      {l: round(float(df[l + "_l"].map(len).mean()), 2) for l in ["Top", "Middle", "Base"]})
print("perfumy bez żadnej nuty:", int(((df.Top_l.map(len) + df.Middle_l.map(len) + df.Base_l.map(len)) == 0).sum()))
print("\nile perfum ma ile akordów:")
print(df["accords_l"].map(len).value_counts().sort_index().to_string())
print("\nnajczęstsze nuty:", all_notes.most_common(10))

przyklad = int(df.index[df.Perfume == "L EAU D ISSEY POUR HOMME NOIR AMBRE"][0])
print("\nprzykład:")
print(df.loc[przyklad, ["Perfume", "Top", "Top_l", "accords_l"]].to_string())

unikalne nuty: 1664
  występujące tylko raz: 343
  występujące co najmniej 3 razy: 1135
  występujące co najmniej 50 razy: 366
unikalne akordy: 84

średnia liczba nut w warstwie: {'Top': 3.11, 'Middle': 3.3, 'Base': 3.5}
perfumy bez żadnej nuty: 0

ile perfum ma ile akordów:
accords_l
1       13
2      101
3      274
4      591
5    23067

najczęstsze nuty: [('musk', 11013), ('bergamot', 8613), ('sandalwood', 8057), ('jasmine', 7741), ('amber', 7725), ('patchouli', 7250), ('vanilla', 6738), ('rose', 6249), ('cedar', 5625), ('mandarin orange', 4006)]

przykład:
Perfume               L EAU D ISSEY POUR HOMME NOIR AMBRE
Top                  saffron, cinnamon, nutmeg, coriander
Top_l              [saffron, cinnamon, nutmeg, coriander]
accords_l    [amber, leather, warm spicy, vanilla, woody]


**Wniosek:** 1664 unikalne nuty, z czego 343 występują tylko raz; próg `MIN_DF_NOTE = 3` odrzuca ok. 530 najrzadszych (1135 nut ma co najmniej 3 wystąpienia; w macierzy cech zostaje 1133, bo tam liczymy, w ilu *perfumach* występuje nuta, a nie ile razy). 84 akordy; 979 perfum ma mniej niż 5 akordów (23 067 ma komplet). W danych nie ma „unknown” i nie ma perfum bez nut.


## 3. Wektory cech (nuty i akordy jako macierze rzadkie)

**Teoria.** Perfumy mają po kilkanaście nut z ponad tysiąca możliwych, więc wektor cech to prawie same zera – trzymamy go jako **macierz rzadką** (zapamiętuje tylko niezerowe elementy).
* **Nuty:** waga warstwy (1/1/1), opcjonalnie podzielona przez `długość_listy^len_exp` (`LENGTH_NORM_EXP = 0` – bez kary za długie piramidy; sprawdzamy to w części 6.1). Nuta powtórzona w kilku warstwach dostaje sumę wag.
* **Akordy:** waga maleje z pozycją (1.0, 0.75, 0.56, 0.42, 0.32).
* **`MIN_DF_NOTE`:** nuty z mniej niż 3 perfum odpadają jako szum.

**Co robi kod:** `to_sparse` liczy w ilu perfumach występuje cecha (`df_cechy`, potrzebne do IDF), zostawia cechy z `df ≥ min_df` i buduje macierz CSR (wiersze = perfumy, kolumny = cechy). Wynik: `X_note`, `X_acc` i nazwy kolumn. Na końcu kontrola, że żadne perfumy nie straciły wszystkich nut.

In [4]:
# Każde perfumy zapisujemy jako rzadki wektor w przestrzeni CECH:
# jedna współrzędna na każdą nutę i jedna na każdy akord.
# Wartość współrzędnej = WAGA tej cechy w tych perfumach (0 = cechy brak).

def note_features(layer_w=None, len_exp=None):
    layer_w = CONFIG["LAYER_WEIGHTS"] if layer_w is None else layer_w
    len_exp = CONFIG["LENGTH_NORM_EXP"] if len_exp is None else len_exp
    wynik = []
    for top, middle, base in zip(df.Top_l, df.Middle_l, df.Base_l):
        cechy = {}
        for lista, w in ((top, layer_w["Top"]),
                         (middle, layer_w["Middle"]),
                         (base, layer_w["Base"])):
            for nuta in lista:
                cechy[nuta] = cechy.get(nuta, 0.0) + w / (len(lista) ** len_exp)
        wynik.append(cechy)
    return wynik

# Akordy są uporządkowane od najbardziej wyczuwalnego, więc waga maleje z pozycją.
def accord_features(rank_w=None):
    wagi = CONFIG["ACCORD_RANK_WEIGHTS"] if rank_w is None else rank_w
    return [{akord: wagi[i] for i, akord in enumerate(lista[:5])}
            for lista in df.accords_l]

# słowniki cech -> macierz rzadka
def to_sparse(cechy_perfum, min_df=1):
    licznik = Counter(k for d in cechy_perfum for k in d)          # w ilu perfumach jest cecha
    vocab = sorted(k for k, v in licznik.items() if v >= min_df)   # zostają cechy z >= min_df perfum
    indeks = {k: i for i, k in enumerate(vocab)}

    wiersze, kolumny, wartosci = [], [], []
    for i, d in enumerate(cechy_perfum):
        for k, v in d.items():
            j = indeks.get(k)
            if j is not None:
                wiersze.append(i); kolumny.append(j); wartosci.append(v)

    X = sp.csr_matrix((wartosci, (wiersze, kolumny)),
                      shape=(len(cechy_perfum), len(vocab)))
    df_cechy = np.array([licznik[k] for k in vocab])
    return X, vocab, df_cechy

X_note, vocab_note, df_note = to_sparse(note_features(), CONFIG["MIN_DF_NOTE"])
X_acc,  vocab_acc,  df_acc  = to_sparse(accord_features())

zero = int((np.asarray(X_note.sum(axis=1)).ravel() == 0).sum())
print("tabela nut:    ", X_note.shape)
print("tabela akordów:", X_acc.shape)
print(f"średnio cech na perfumy: nuty {X_note.nnz / N:.1f}, akordy {X_acc.nnz / N:.1f}")
print(f"wypełnienie tabeli nut: {X_note.nnz / (X_note.shape[0] * X_note.shape[1]):.2%}")
print("perfumy z ZEROWYM wektorem nut po filtrze MIN_DF:", zero, "(powinno być 0)")
assert zero == 0, "zmniejsz MIN_DF_NOTE"

pokaz = lambda X, vocab, i: {vocab[j]: round(float(v), 2) for j, v in zip(X[i].indices, X[i].data)}
print(f"\n{df.Perfume[przyklad]}, nuty:  ", pokaz(X_note, vocab_note, przyklad))
print(f"{df.Perfume[przyklad]}, akordy:", pokaz(X_acc, vocab_acc, przyklad))
print("\nnajczęstsze nuty (w ilu perfumach):", [(vocab_note[j], int(df_note[j])) for j in np.argsort(-df_note)[:5]])
print("najczęstsze akordy (w ilu perfumach):", [(vocab_acc[j], int(df_acc[j])) for j in np.argsort(-df_acc)[:5]])

tabela nut:     (24046, 1133)
tabela akordów: (24046, 84)
średnio cech na perfumy: nuty 9.8, akordy 4.9
wypełnienie tabeli nut: 0.87%
perfumy z ZEROWYM wektorem nut po filtrze MIN_DF: 0 (powinno być 0)

L EAU D ISSEY POUR HOMME NOIR AMBRE, nuty:   {'amber': 1.0, 'cedar': 1.0, 'cinnamon': 1.0, 'coriander': 1.0, 'leather': 1.0, 'nutmeg': 1.0, 'papyrus': 1.0, 'saffron': 1.0, 'tonka bean': 1.0, 'vanilla': 1.0, 'vetiver': 1.0}
L EAU D ISSEY POUR HOMME NOIR AMBRE, akordy: {'amber': 1.0, 'leather': 0.75, 'vanilla': 0.42, 'warm spicy': 0.56, 'woody': 0.32}

najczęstsze nuty (w ilu perfumach): [('musk', 10942), ('bergamot', 8608), ('sandalwood', 8007), ('amber', 7681), ('jasmine', 7669)]
najczęstsze akordy (w ilu perfumach): [('woody', 12523), ('citrus', 9483), ('aromatic', 7620), ('sweet', 7471), ('fruity', 6928)]


**Wniosek:** macierz nut ma 1133 kolumny, macierz akordów 84; średnio 9,8 nuty i 4,9 akordu na perfumy, wypełnienie tabeli nut to 0,87%. Żadne perfumy nie straciły wszystkich nut po filtrze.


## 4. Funkcje modeli: metoda A, metoda B / B\* i hybryda

Te same funkcje (pod tymi samymi nazwami) są w `embeddings_v3_proste` – tam bez parametrów badawczych. Tu mają dodatkowe przełączniki, żeby ablacje zmieniały jedną rzecz naraz. Funkcje **biorą macierze jako argumenty** (nie zmienne globalne) – potrzebne do uczciwej oceny z ukrytymi akordami (część 5).

### Metoda A – TF-IDF + SVD
1. **IDF** (`IDF = ln((1+N)/(1+df)) + 1`, waga `IDF^IDF_POWER`): pospolite cechy (piżmo jest w połowie perfum) mówią mało o *różnicy* między perfumami.
2. **Dwa bloki** (nuty, akordy) skracamy osobno do długości 1 i sklejamy `[√α·akordy, √(1−α)·nuty]`. Po skróceniu iloczyn takich wektorów to dokładnie `α·cos(akordy) + (1−α)·cos(nuty)` – α jest dosłownie udziałem akordów (sprawdzimy to w 6.2).
3. **SVD** kompresuje ok. 1100 wymiarów do 256: podobne nuty (np. różne cytrusy) wpływają na te same osie.

### Metoda B – wektory nut i akordów uczone ze współwystępowania
1. Nuty (`n:vanilla`) i akordy (`a:woody`) to „słowa”, perfumy to „zdania”. **B\*** dokłada słowa `a1:<akord>` dla akordu nr 1.
2. Macierz współwystępowania `C[i,j]` (w ilu perfumach razem) → **PPMI**: `log P(i,j) / (P(i)·P(j))` tylko dodatnie; rozkład kontekstów `P(j)` wygładzony potęgą `B_CTX_SMOOTH`. **B\*** odejmuje od PMI `B_PPMI_SHIFT` przed obcięciem – wycina słabe, przypadkowe powiązania.
3. **SVD** tej tabeli daje wektory tokenów; **wektor perfum = średnia wektorów jej tokenów** (akordy × `B_ACCORD_GAIN`), potem usuwamy wspólny kierunek wszystkich perfum (zawyżałby cosinusy).

### Hybryda
`hybrid(A, B, w)` skleja `[√w·A, √(1−w)·B]`: podobieństwo = `w·cos_A + (1−w)·cos_B`.

**Parametry badawcze:** `alpha`, `dim`, `idf_power`, `exact` (A bez SVD); `gain`, `use_accords`, `eig` (potęga „siły osi”), `remove_pc`, `top1_weight`, `ppmi_shift` (B). `Xa=None` buduje model tylko z nut.

In [5]:
def idf_vec(X, power):
    """IDF = ln((1+n)/(1+df)) + 1 policzony z macierzy X; waga = IDF**power."""
    dfc = np.asarray((X > 0).sum(axis=0)).ravel()
    return (np.log((1 + X.shape[0]) / (1 + dfc)) + 1.0) ** power


def build_A(Xn, Xa, alpha=None, dim=None, idf_power=None, exact=False):
    """Metoda A: TF-IDF nut i akordów -> sklejenie z udziałem alpha -> SVD -> długość 1.
    Xa=None albo alpha=0 -> model tylko z nut; exact=True -> bez SVD (rzadkie wektory)."""
    alpha = CONFIG["A_ALPHA_ACCORD"] if alpha is None else alpha
    dim = CONFIG["DIM"] if dim is None else dim
    idf_power = CONFIG["IDF_POWER"] if idf_power is None else idf_power
    Sn = normalize(Xn.multiply(idf_vec(Xn, idf_power)).tocsr())
    if Xa is not None and alpha > 0:
        Sa = normalize(Xa.multiply(idf_vec(Xa, idf_power)).tocsr())
        V = sp.hstack([np.sqrt(alpha) * Sa, np.sqrt(1 - alpha) * Sn]).tocsr()
    else:
        V = Sn
    if exact:
        return normalize(V)
    Z = TruncatedSVD(n_components=dim, random_state=CONFIG["SEED"]).fit_transform(V)
    return normalize(Z).astype(np.float32)


def build_B(Xn, Xa, dim=None, ctx_smooth=None, gain=None, use_accords=True, eig=0.0, remove_pc=True,
            top1_weight=None, ppmi_shift=None):
    """Metoda B: współwystępowanie tokenów -> PPMI -> SVD -> średnia wektorów tokenów.
    top1_weight=0 i ppmi_shift=0 -> starsza wersja B; domyślnie (z CONFIG) -> B*.
    Zwraca (wektory perfum, wektory tokenów, nazwy tokenów)."""
    dim = CONFIG["DIM"] if dim is None else dim
    ctx_smooth = CONFIG["B_CTX_SMOOTH"] if ctx_smooth is None else ctx_smooth
    gain = CONFIG["B_ACCORD_GAIN"] if gain is None else gain
    top1_weight = CONFIG["B_TOP1_WEIGHT"] if top1_weight is None else top1_weight
    ppmi_shift = CONFIG["B_PPMI_SHIFT"] if ppmi_shift is None else ppmi_shift
    seed = CONFIG["SEED"]

    bloki, nazwy = [Xn], ["n:" + v for v in vocab_note]
    if use_accords and Xa is not None:
        bloki.append(Xa * gain);  nazwy += ["a:" + v for v in vocab_acc]
        if top1_weight > 0:                                   # tokeny "akord nr 1"
            bloki.append((Xa >= CONFIG["ACCORD_RANK_WEIGHTS"][0] - 1e-9).astype(float) * top1_weight)
            nazwy += ["a1:" + v for v in vocab_acc]
    W = sp.hstack(bloki).tocsr()                              # wagi tokenów w perfumach

    Wb = W.copy(); Wb.data[:] = 1                             # współwystępowanie liczy samą OBECNOŚĆ
    C = (Wb.T @ Wb).toarray().astype(np.float64)              # C[i, j] = w ilu perfumach razem i oraz j
    np.fill_diagonal(C, 0)                                    # token nie współwystępuje sam ze sobą
    suma = C.sum()
    p_i = C.sum(axis=1) / suma
    p_j = C.sum(axis=0) ** ctx_smooth                         # wygładzenie: potęga < 1
    p_j = p_j / p_j.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log((C / suma) / (p_i[:, None] * p_j[None, :])) - ppmi_shift
    ppmi = np.where(np.isfinite(pmi) & (pmi > 0), pmi, 0.0)   # tylko wartości dodatnie

    U, s, _ = randomized_svd(sp.csr_matrix(ppmi), n_components=dim, random_state=seed)
    E = normalize(U * (s ** eig))                             # wektory tokenów (eig=0: samo U)
    P = normalize(normalize(W) @ E)                           # wektor perfum = średnia wektorów tokenów
    if remove_pc:
        u = TruncatedSVD(1, random_state=seed).fit(P).components_[0]
        P = normalize(P - np.outer(P @ u, u))                 # usuń wspólny kierunek
    return P.astype(np.float32), E, nazwy


def hybrid(Za, Zb, w_A):
    """Sklejenie wektorów A i B: iloczyn = w_A*cos_A + (1-w_A)*cos_B."""
    return normalize(np.hstack([np.sqrt(w_A) * Za, np.sqrt(1 - w_A) * Zb])).astype(np.float32)

print("funkcje modeli zdefiniowane")

funkcje modeli zdefiniowane

## 5. Jak uczciwie mierzyć jakość (trzy miary)

**Problem.** Nie mamy etykiet „te perfumy są podobne”. Jedyny sygnał o *odbieranym* zapachu to **akordy** (głosy użytkowników). Pytamy więc: *czy sąsiedzi perfum mają podobne akordy?* – cosinus wektorów akordów, uśredniony po 10 najbliższych sąsiadach (**zgodność akordów**, dawniej AccAgree@10).

**Pułapka (wyciek).** Jeśli akordy wchodzą do wektora perfum, test sprawdza tylko, czy model zapamiętał to, co mu wpisano. Rozwiązania:
* **Model tylko z nut** (`eval_notes`) – akordy są wtedy niezależną odpowiedzią; uczciwe bez dodatkowych zabiegów.
* **Zimny start** (`eval_cold`) dla modeli używających akordów: perfumy dzielimy na 5 części; dla każdej części **usuwamy akordy jej perfum jeszcze przed dopasowaniem modelu**, pytamy o ich sąsiadów (znamy tylko nuty), kandydatami są pozostałe perfumy, a ocena idzie po prawdziwych akordach.

**Dwie miary pomocnicze** liczone na modelu dopasowanym na całości (`diag_full`): **bracia w top-10** i **skośność hubness** (ile razy każde perfumy trafiają do cudzych list top-10; wysoka skośność = kilka „hubów” zaśmieca listy; liczymy też odsetek „anty-hubów”, które nie trafiają nigdzie).

**Statystyka.** Zgodność liczymy na każde zapytanie osobno, więc możemy podać błąd standardowy (SE = odchylenie/√n) i **różnicę parowaną** dwóch modeli (te same zapytania w obu; SE różnicy jest dużo mniejszy). Różnice mniejsze niż 2–3 SE traktuj jako szum.

**Co robi kod:** `knn_top` liczy cosinusy porcji zapytań do wszystkich kandydatów (iloczyn macierzy), wyklucza zapytanie jako własnego sąsiada i wybiera 10 najbliższych szybkim `argpartition`; **remisy rozstrzyga deterministycznie** (stary kod rozstrzygał losowo – to dawało szum ±0,001). `Q_SUB` to stała próbka zapytań do ablacji (`N_SUBSAMPLE`).

In [6]:
X_acc_target, _, _ = to_sparse(accord_features(rank_w=[1.0, 0.75, 0.56, 0.42, 0.32]))   # "odpowiedzi": pełne akordy, STAŁE wagi
Y = normalize(X_acc_target).toarray().astype(np.float32)
FLANK = pd.factorize(klucz)[0]                                          # klucz "braci"
LOSOWO = float(Y.mean(0) @ Y.mean(0))                                   # poziom losowy zgodności akordów
Q_SUB = np.sort(np.random.default_rng(CONFIG["SEED"]).choice(N, min(CONFIG["N_SUBSAMPLE"], N), replace=False))


def knn_top(Zq, Zdb, q, k=10, kandydaci=None, chunk=1000):
    """k najbliższych sąsiadów (cosinus); remisy: niższy indeks wygrywa (wynik powtarzalny)."""
    dens = lambda M: (M.toarray() if sp.issparse(M) else np.asarray(M)).astype(np.float32)
    Zq, Zdb = dens(Zq), dens(Zdb)
    out = np.zeros((len(q), k), dtype=np.int64)
    for s in range(0, len(q), chunk):
        qi = q[s:s + chunk]
        sim = Zq[s:s + chunk] @ Zdb.T
        if kandydaci is not None:
            sim[:, ~kandydaci] = -9
        sim[np.arange(len(qi)), qi] = -9                                 # perfumy nie są własnym sąsiadem
        czesc = np.argpartition(-sim, k + 40, axis=1)[:, :k + 40]
        czesc.sort(axis=1)
        kol = np.argsort(-np.take_along_axis(sim, czesc, axis=1), axis=1, kind="stable")[:, :k]
        out[s:s + chunk] = np.take_along_axis(czesc, kol, axis=1)
    return out


def zgodnosc(top, q):
    """Zgodność akordów na zapytanie: średni cosinus akordów zapytania i jego sąsiadów."""
    return np.einsum("qkd,qd->qk", Y[top], Y[q]).mean(1)


def eval_notes(Z, q=None):
    """Model zbudowany WYŁĄCZNIE z nut (akordy są niezależną odpowiedzią) - uczciwe bez zimnego startu."""
    q = Q_SUB if q is None else q
    return zgodnosc(knn_top(Z[q], Z, q), q)


def eval_cold(model_fn, n_folds=5, per_fold=None):
    """ZIMNY START: akordy perfum z foldu H usuwane PRZED dopasowaniem modelu; zapytania = H, kandydaci = reszta.
    model_fn(X_note, X_acc_zamaskowane) -> wektory perfum."""
    fold = np.random.default_rng(CONFIG["SEED"]).permutation(N) % n_folds
    rng_sub = np.random.default_rng(CONFIG["SEED"] + 1)
    czesci = []
    for f in range(n_folds):
        H = fold == f
        Xa_ukryte = (sp.diags((~H).astype(float)) @ X_acc).tocsr()
        Z = model_fn(X_note, Xa_ukryte)
        q = np.where(H)[0]
        if per_fold is not None and per_fold < len(q):
            q = np.sort(rng_sub.choice(q, per_fold, replace=False))
        czesci.append(zgodnosc(knn_top(Z[q], Z, q, kandydaci=~H), q))
    return np.concatenate(czesci)


def diag_full(Z):
    """Bracia w top-10 i hubness - na modelu dopasowanym na całości, wszystkie perfumy jako zapytania."""
    q = np.arange(N)
    top = knn_top(Z, Z, q)
    grp = np.bincount(FLANK)[FLANK]; ok = grp > 1
    bracia = float(((FLANK[top] == FLANK[:, None]).sum(1)[ok] / np.minimum(10, grp[ok] - 1)).mean())
    nk = np.bincount(top.ravel(), minlength=N)
    return dict(bracia=round(bracia, 4), hubness=round(float(skew(nk)), 2), anty_huby=round(float((nk == 0).mean()), 3))


def pm(v):
    """średnia ± błąd standardowy"""
    return f"{v.mean():.4f} ± {v.std(ddof=1) / np.sqrt(len(v)):.4f}"


def roznica(a, b):
    """różnica parowana a-b: średnia ± SE (te same zapytania)"""
    d = a - b
    return f"{d.mean():+.4f} ± {d.std(ddof=1) / np.sqrt(len(d)):.4f}"

print(f"poziom losowy zgodności akordów: {LOSOWO:.4f} | zapytań w ablacjach: {len(Q_SUB)}")

poziom losowy zgodności akordów: 0.2317 | zapytań w ablacjach: 8000

### 5.1 Kontrola samej miary i demonstracja wycieku

**Cel:** zanim zaufamy liczbom, sprawdzamy trzy rzeczy.
1. **Granice:** wektory losowe powinny dać wynik ≈ poziomowi losowemu, a same wektory akordów (sufit dla modeli opartych na akordach) – ok. 0,92.
2. **Powtarzalność:** drugie uruchomienie daje identyczne listy sąsiadów.
3. **Wyciek:** ten sam model A (nuty + akordy) oceniony „po staremu” (akordy zapytania w wektorze) i w zimnym starcie. Różnica pokazuje, dlaczego dla modeli z akordami nie wolno używać zwykłej miary.

In [7]:
A_notes = build_A(X_note, None)                                       # A tylko z nut
losowe = normalize(np.random.default_rng(0).normal(size=(N, 64)))

print("poziom losowy (wzór):                        ", round(LOSOWO, 4))
print("wektory losowe:                              ", pm(eval_notes(losowe)))
print("sufit: same wektory akordów:                 ", pm(eval_notes(Y)))
print("A tylko z nut:                               ", pm(eval_notes(A_notes)))

t1 = knn_top(A_notes[Q_SUB], A_notes, Q_SUB); t2 = knn_top(A_notes[Q_SUB], A_notes, Q_SUB)
print("listy sąsiadów identyczne przy ponownym uruchomieniu:", bool((t1 == t2).all()))

A_pelny = build_A(X_note, X_acc)
print("\nA nuty+akordy, miara 'po staremu' (WYCIEK):   ", pm(eval_notes(A_pelny)))
print("A nuty+akordy, zimny start (uczciwie):        ", pm(eval_cold(lambda Xn, Xa: build_A(Xn, Xa), per_fold=1600)))

poziom losowy (wzór):                         0.2317
wektory losowe:                               0.2320 ± 0.0010
sufit: same wektory akordów:                  0.9177 ± 0.0006
A tylko z nut:                                0.5241 ± 0.0016
listy sąsiadów identyczne przy ponownym uruchomieniu: True

A nuty+akordy, miara 'po staremu' (WYCIEK):    0.8189 ± 0.0008
A nuty+akordy, zimny start (uczciwie):         0.5193 ± 0.0016


**Wniosek:** wektory losowe dają 0,232 (poziom losowy), a sufit dla modeli opartych na akordach to 0,918. A z samych nut: 0,524. Ten sam model A z akordami daje **0,819 w mierze „po staremu” i tylko 0,519 w zimnym starcie** – różnica 0,30 to czysty wyciek. Dlatego dla modeli używających akordów oceniamy wyłącznie zimnym startem. Listy sąsiadów są powtarzalne.


## 6. Ablacje: które ustawienia mają znaczenie

W każdym teście **zmieniamy jedną rzecz naraz** i podajemy różnicę parową ± SE względem ustawienia z `CONFIG`.

### 6.1 Cechy nut: waga warstw, normalizacja długości, moc IDF (27 wariantów)

**Pytania:** (1) czy baza lub głowa powinny ważyć więcej, (2) czy karać długie piramidy (`len_exp`), (3) jak mocno osłabiać pospolite nuty (`idf_power`).
Model: tylko nuty, cosinus na pełnych rzadkich wektorach (bez SVD) – wynik zależy wyłącznie od wag cech. **Wagi warstw 1/1/1 są ustaloną decyzją**; ta ablacja tylko ją dokumentuje (większa waga bazy wypada gorzej).

In [8]:
warianty_warstw = {
    "płaskie 1/1/1":              {"Top": 1.0, "Middle": 1.0, "Base": 1.0},
    "baza ważniejsza 0.5/1/1.2":  {"Top": 0.5, "Middle": 1.0, "Base": 1.2},
    "głowa ważniejsza 1.2/1/0.8": {"Top": 1.2, "Middle": 1.0, "Base": 0.8},
}

wyniki_abl, t0 = {}, time.time()
for nazwa, lw in warianty_warstw.items():
    for le in [0.0, 0.5, 1.0]:
        Xn, _, _ = to_sparse(note_features(lw, le), CONFIG["MIN_DF_NOTE"])
        for p in [0.0, 0.5, 1.0]:
            wyniki_abl[(nazwa, le, p)] = eval_notes(build_A(Xn, None, idf_power=p, exact=True))
print(f"czas: {time.time() - t0:.0f} s")

ref = wyniki_abl[("płaskie 1/1/1", 0.0, 0.5)]                            # ustawienie z CONFIG
abl = pd.DataFrame([{"warstwy": k[0], "len_exp": k[1], "idf_power": k[2], "zgodność": v.mean(),
                     "różnica vs CONFIG": (v - ref).mean(), "SE różnicy": (v - ref).std(ddof=1) / np.sqrt(len(v))}
                    for k, v in wyniki_abl.items()]).sort_values("zgodność", ascending=False).reset_index(drop=True)
print(f"\npoziom losowy: {LOSOWO:.4f}\n")
display(abl.round(4))

for czynnik in ["warstwy", "len_exp", "idf_power"]:
    print(f"\n wpływ czynnika: {czynnik}")
    display(abl.groupby(czynnik)["zgodność"].agg(["mean", "max"]).round(4))

czas: 140 s

poziom losowy: 0.2317

                       warstwy  len_exp  idf_power  zgodność  różnica vs CONFIG  SE różnicy
0                płaskie 1/1/1      0.0        0.5    0.5278             0.0000      0.0000
1   głowa ważniejsza 1.2/1/0.8      0.0        0.5    0.5243            -0.0035      0.0008
2   głowa ważniejsza 1.2/1/0.8      0.0        0.0    0.5179            -0.0099      0.0009
3                płaskie 1/1/1      0.5        0.5    0.5149            -0.0129      0.0007
4                płaskie 1/1/1      0.0        0.0    0.5125            -0.0153      0.0008
5   głowa ważniejsza 1.2/1/0.8      0.5        0.5    0.5087            -0.0191      0.0010
6                płaskie 1/1/1      0.0        1.0    0.5062            -0.0217      0.0009
7                płaskie 1/1/1      0.5        0.0    0.5043            -0.0236      0.0009
8   głowa ważniejsza 1.2/1/0.8      0.5        0.0    0.5036            -0.0242      0.0010
9   głowa ważniejsza 1.2/1/0.8      0.0     


**Wniosek:** najlepsze jest ustawienie z `CONFIG` (płaskie warstwy, `len_exp = 0`, `idf_power = 0,5`; zgodność 0,5278).
* **Warstwy:** „baza ważniejsza” jest wyraźnie gorsza (średnio 0,461 vs 0,499 dla płaskich; najlepszy wariant −0,050), „głowa ważniejsza” minimalnie gorsza (−0,0035 ± 0,0008). Potwierdza to ustaloną decyzję 1/1/1.
* **`len_exp`:** średnia 0,4995 (0) → 0,4897 (0,5) → 0,4670 (1) – kara za długie piramidy szkodzi.
* **`idf_power`:** 0,5 jest najlepsze (średnia 0,4945), brak osłabiania 0,4844, pełne 0,4772.


### 6.2 Metoda A: kontrola działania i liczba wymiarów

**Co sprawdzamy:**
1. **Czy α naprawdę jest udziałem akordów?** Dla losowej pary perfum: cosinus po sklejeniu bloków musi równać się `α·cos(akordy) + (1−α)·cos(nuty)`. SVD trochę zmienia wartość (kompresja stratna).
2. **Retencja:** ile z 10 najbliższych sąsiadów wg dokładnego cosinusa (bez SVD) zostaje w top-10 po kompresji. Niewielka utrata jest normalna – sąsiedzi o prawie jednakowych cosinusach zamieniają się miejscami.
3. **Liczba wymiarów (`DIM`):** model tylko z nut, 32–512 osi oraz wersja bez kompresji. Za mało osi gubi informację, za dużo nic nie daje.

In [9]:
alpha, p = CONFIG["A_ALPHA_ACCORD"], CONFIG["IDF_POWER"]
Sn = normalize(X_note.multiply(idf_vec(X_note, p)).tocsr())
Sa = normalize(X_acc.multiply(idf_vec(X_acc, p)).tocsr())
V_A = sp.hstack([np.sqrt(alpha) * Sa, np.sqrt(1 - alpha) * Sn]).tocsr()
Z_A = build_A(X_note, X_acc)

svd = TruncatedSVD(CONFIG["DIM"], random_state=CONFIG["SEED"]).fit(V_A)
print(f"1. kształt przed kompresją {V_A.shape}, po SVD {Z_A.shape}; wariancja wyjaśniona przez {CONFIG['DIM']} osi: "
      f"{svd.explained_variance_ratio_.sum():.1%}")

i, j = np.random.default_rng(CONFIG["SEED"]).choice(N, 2, replace=False)
lewa  = (V_A[i] @ V_A[j].T).toarray()[0, 0]
prawa = alpha * (Sa[i] @ Sa[j].T).toarray()[0, 0] + (1 - alpha) * (Sn[i] @ Sn[j].T).toarray()[0, 0]
print(f"   cos po sklejeniu = {lewa:.4f}; alpha*cos(akordy)+(1-alpha)*cos(nuty) = {prawa:.4f}; po SVD = {float(Z_A[i] @ Z_A[j]):.4f}")

q = Q_SUB[:1000]
V_n = build_A(X_note, X_acc, exact=True)
retencja = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(knn_top(V_n[q], V_n, q), knn_top(Z_A[q], Z_A, q))])
print(f"2. retencja top-10 po kompresji SVD względem dokładnego cosinusa: {retencja:.1%}")

print("\n3. liczba wymiarów (A tylko z nut), różnica parowana vs DIM=256:")
wyn_dim = {d: eval_notes(build_A(X_note, None, dim=d)) for d in [32, 64, 128, 256, 384, 512]}
wyn_dim["bez SVD"] = eval_notes(build_A(X_note, None, exact=True))
for d, v in wyn_dim.items():
    print(f"   {str(d):8s} zgodność {pm(v)}   różnica vs 256: {roznica(v, wyn_dim[256])}")

1. kształt przed kompresją (24046, 1217), po SVD (24046, 256); wariancja wyjaśniona przez 256 osi: 88.2%
   cos po sklejeniu = 0.0893; alpha*cos(akordy)+(1-alpha)*cos(nuty) = 0.0893; po SVD = 0.1043
2. retencja top-10 po kompresji SVD względem dokładnego cosinusa: 64.0%

3. liczba wymiarów (A tylko z nut), różnica parowana vs DIM=256:
   32       zgodność 0.4502 ± 0.0018   różnica vs 256: -0.0738 ± 0.0014
   64       zgodność 0.4816 ± 0.0018   różnica vs 256: -0.0425 ± 0.0012
   128      zgodność 0.5090 ± 0.0017   różnica vs 256: -0.0150 ± 0.0008
   256      zgodność 0.5241 ± 0.0016   różnica vs 256: +0.0000 ± 0.0000
   384      zgodność 0.5274 ± 0.0016   różnica vs 256: +0.0034 ± 0.0005
   512      zgodność 0.5276 ± 0.0016   różnica vs 256: +0.0035 ± 0.0006
   bez SVD  zgodność 0.5278 ± 0.0016   różnica vs 256: +0.0037 ± 0.0007


**Wniosek:** (1) α jest dosłownie udziałem akordów (0,0893 = 0,0893). 256 osi wyjaśnia 88,2% wariancji. (2) Retencja top-10 po kompresji to 64% – sąsiedzi o niemal identycznych cosinusach zamieniają się miejscami; sama zgodność akordów spada przez kompresję tylko o 0,004. (3) Poniżej 256 osi wyraźnie tracimy (128: −0,015, 64: −0,043, 32: −0,074); powyżej zysk jest mały (384/512: +0,003 ± 0,0006; wersja bez SVD +0,004). `DIM = 256` zostaje.


### 6.3 Metoda B: czy wektory tokenów mają sens?

**Cel:** obejrzeć najbliższych sąsiadów wybranych nut i akordów. Jeśli metoda działa, „vanilla” sąsiaduje z nutami słodkimi, „lavender” z ziołami, `a:aquatic` z `a:ozonic` i owocami wodnymi. W B\* pojawiają się też tokeny `a1:…` (akord nr 1) jako sąsiedzi nut, które zwykle go otwierają.

Dodatkowo: średni cosinus losowych par perfum (powinien być bliski 0 po usunięciu wspólnego kierunku) i cosinus dwóch wskazanych po nazwie perfum.

**Obserwacja warta uwagi:** jeśli najbliższym sąsiadem jakiejś nuty jest jej wariant lub synonim (tu: `n:oud` → `n:agarwood`, `n:rose` → `n:bulgarian rose`), to kandydat do przyszłego scalania nut w bazie. Model sam podpowiada, które nuty szczegółowe warto potem uogólnić.

In [10]:
Z_B0, E_B0, nazwy_B0 = build_B(X_note, X_acc, top1_weight=0.0, ppmi_shift=0.0)     # starsza wersja B
Z_Bs, E_Bs, nazwy_Bs = build_B(X_note, X_acc)                                      # B* (CONFIG)
print(f"B (starsza): tokenów {len(nazwy_B0)}, wektory perfum {Z_B0.shape}")
print(f"B*         : tokenów {len(nazwy_Bs)} (w tym 'akord nr 1'), wektory perfum {Z_Bs.shape}")

def nearest_tokens(E, nazwy, token, k=6):
    idx = {t: i for i, t in enumerate(nazwy)}
    sims = E @ E[idx[token]]
    return ", ".join(f"{nazwy[j]} ({sims[j]:.2f})" for j in np.argsort(-sims)[1:k + 1])

print("\nnajbliższe tokeny (B starsza):")
for t in ["n:vanilla", "n:tonka bean", "n:oud", "n:lavender", "n:rose", "n:vetiver", "a:aquatic", "a:leather"]:
    if t in nazwy_B0:
        print(f"  {t:13s} -> {nearest_tokens(E_B0, nazwy_B0, t)}")
print("\nnajbliższe tokeny (B*):")
for t in ["n:leather", "a1:leather", "a1:aquatic"]:
    print(f"  {t:13s} -> {nearest_tokens(E_Bs, nazwy_Bs, t)}")

r = np.random.default_rng(0)
pi, pj = r.integers(0, N, 100000), r.integers(0, N, 100000)
c = np.einsum("ij,ij->i", Z_B0[pi], Z_B0[pj])
print(f"\nlosowe pary perfum (B): średni cos {c.mean():.3f}, odchylenie {c.std():.3f}")
a_, b_ = [int(df.index[df.Perfume == n][0]) for n in ["BABYLON", "L EAU D ISSEY POUR HOMME NOIR AMBRE"]]
print(f"cos pary ({df.Perfume[a_]} | {df.Perfume[b_]}): B {float(Z_B0[a_] @ Z_B0[b_]):.4f}, B* {float(Z_Bs[a_] @ Z_Bs[b_]):.4f}")

B (starsza): tokenów 1217, wektory perfum (24046, 256)
B*         : tokenów 1301 (w tym 'akord nr 1'), wektory perfum (24046, 256)

najbliższe tokeny (B starsza):
  n:vanilla     -> n:sandalwood (0.63), a:vanilla (0.61), n:heliotrope (0.59), n:plum (0.58), n:musk (0.57), n:amber (0.56)
  n:tonka bean  -> n:bergamot (0.57), n:pepper (0.56), n:sandalwood (0.55), n:amber (0.55), n:cedar (0.55), n:benzoin (0.55)
  n:oud         -> n:agarwood (0.35), n:cedarwood (0.32), n:black cherry (0.30), n:frankincense (0.29), n:smoke (0.27), a:leather (0.27)
  n:lavender    -> n:rosemary (0.73), n:geranium (0.72), n:basil (0.71), a:aromatic (0.71), a:lavender (0.71), a:fresh spicy (0.70)
  n:rose        -> n:musk (0.63), n:sandalwood (0.62), n:jasmine (0.61), n:amber (0.60), n:lily-of-the-valley (0.57), n:bulgarian rose (0.56)
  n:vetiver     -> n:coriander (0.63), n:cedar (0.61), n:oakmoss (0.59), a:woody (0.59), n:geranium (0.59), n:pepper (0.58)
  a:aquatic     -> a:ozonic (0.63), n:watermelon (0.4


**Wniosek:** sąsiedztwa tokenów mają sens: `n:lavender` leży przy ziołach (`rosemary`, `basil`) i `a:lavender`, `a:aquatic` przy `a:ozonic` i owocach wodnych, `n:leather` przy `a:leather`, `birch`, `suede`. W B\* tokeny `a1:…` stoją tuż przy odpowiednich akordach (np. `a1:leather` ↔ `a:leather` 0,82). Losowe pary perfum mają średni cosinus ≈ 0 (−0,001), czyli usunięcie wspólnego kierunku zadziałało.


### 6.4 Ulepszenie B → B\*: token „akord nr 1” i przesunięty PPMI

**Problem:** w B współwystępowanie liczy tylko *obecność* akordu – akord na 1. i na 5. miejscu wchodzi tak samo, więc przestrzeń tokenów nie „wie”, który akord dominuje.
* **Token „akord nr 1”** (`top1_weight`): dla każdych perfum dokładamy słowo `a1:<akord nr 1>`. Nuty uczą się, z jakim *dominującym* akordem współwystępują.
* **Przesunięty PPMI** (`ppmi_shift`): `max(PMI − 0,69, 0)`. PMI zawyża znaczenie rzadkich, przypadkowych par; przesunięcie je odcina. (W literaturze to *shifted PPMI*; ln 2 odpowiada dwóm „negatywnym próbkom”.)

**Ocena:** zimny start (model używa akordów, więc zwykła miara byłaby okrężna) + bracia i hubness z modelu dopasowanego na całości. Różnice parowane względem B starszej.

In [11]:
warianty_B = {
    "B (starsza)":                    dict(top1_weight=0.0,  ppmi_shift=0.0),
    "B + token 'akord nr 1'":         dict(top1_weight=0.75, ppmi_shift=0.0),
    "B + przesunięty PPMI":           dict(top1_weight=0.0,  ppmi_shift=0.69),
    "B* = oba":                       dict(top1_weight=0.75, ppmi_shift=0.69),
}
zimne_B, tab = {}, []
for nazwa, kw in warianty_B.items():
    zimne_B[nazwa] = eval_cold(lambda Xn, Xa, kw=kw: build_B(Xn, Xa, **kw)[0])
    d = diag_full(build_B(X_note, X_acc, **kw)[0])
    tab.append({"model": nazwa, "zgodność (zimny start)": zimne_B[nazwa].mean(), "SE": zimne_B[nazwa].std(ddof=1) / np.sqrt(N), **d})
display(pd.DataFrame(tab).round(4))
print("różnice parowane zgodności względem B (starszej):")
for nazwa in list(warianty_B)[1:]:
    print(f"  {nazwa:26s} {roznica(zimne_B[nazwa], zimne_B['B (starsza)'])}")

                    model  zgodność (zimny start)      SE  bracia  hubness  anty_huby
0             B (starsza)                  0.5831  0.0011  0.1283     2.90      0.103
1  B + token 'akord nr 1'                  0.6107  0.0011  0.1194     2.84      0.096
2    B + przesunięty PPMI                  0.5936  0.0011  0.1176     2.57      0.100
3                B* = oba                  0.6336  0.0011  0.1120     2.48      0.089
różnice parowane zgodności względem B (starszej):
  B + token 'akord nr 1'     +0.0276 ± 0.0003
  B + przesunięty PPMI       +0.0105 ± 0.0005
  B* = oba                   +0.0504 ± 0.0005


**Wniosek:** oba ulepszenia działają, a razem się sumują: token „akord nr 1” **+0,0276 ± 0,0003**, przesunięty PPMI **+0,0105 ± 0,0005**, B\* **+0,0504 ± 0,0005** (0,5831 → 0,6336). Skutki uboczne: bracia w top-10 spadają (0,128 → 0,112), za to hubness maleje (2,90 → 2,48). Spadek braci naprawia hybryda z A (część 7).


### 6.5 Pozostałe parametry B (jedna zmiana naraz względem B\*)

**Cel:** w starej ablacji parametry B oceniano na samych nutach; tu sprawdzamy je **w pełnym modelu** (z akordami) w zimnym starcie, bo wnioski z „tylko nut” nie zawsze się przenoszą.
* `ctx_smooth` – wygładzenie rozkładu kontekstów w PPMI (niższe = silniejsze osłabianie bardzo częstych tokenów).
* `eig` – czy wektory tokenów skalować „siłą osi” (0 = same wektory `U`, 0,5 = pierwiastek).
* `remove_pc` – usunięcie wspólnego kierunku z wektorów perfum.

Zimny start na podpróbie (`per_fold=1600`).

In [12]:
ref_B = eval_cold(lambda Xn, Xa: build_B(Xn, Xa)[0], per_fold=1600)
print("B* (CONFIG):", pm(ref_B))
inne = {"ctx_smooth = 0.25": dict(ctx_smooth=0.25), "ctx_smooth = 0.75": dict(ctx_smooth=0.75),
        "eig = 0.5": dict(eig=0.5), "remove_pc = False": dict(remove_pc=False)}
for nazwa, kw in inne.items():
    v = eval_cold(lambda Xn, Xa, kw=kw: build_B(Xn, Xa, **kw)[0], per_fold=1600)
    print(f"  {nazwa:20s} różnica vs B*: {roznica(v, ref_B)}")

B* (CONFIG): 0.6348 ± 0.0018
  ctx_smooth = 0.25    różnica vs B*: -0.0360 ± 0.0008
  ctx_smooth = 0.75    różnica vs B*: -0.0103 ± 0.0012
  eig = 0.5            różnica vs B*: +0.0053 ± 0.0004
  remove_pc = False    różnica vs B*: -0.0153 ± 0.0007


**Wniosek:** parametry z `CONFIG` są dobre. `ctx_smooth = 0,25` pogarsza o 0,036 (w modelu z samych nut wcześniej poprawiał – wnioski z „tylko nut” się nie przenoszą), `0,75` o 0,010, brak usunięcia wspólnego kierunku o 0,015. Jedyna zmiana na plus to `eig = 0,5` (+0,0053 ± 0,0004). Nie włączam jej: zysk jest mały, dobrany na tym samym zbiorze, a w osobnym pomiarze zmniejszała braci; to kandydat do sprawdzenia na `pairs_test.csv`.


### 6.6 Dlaczego udziału akordów (`alpha`, `gain`) nie da się wybrać miarą opartą na akordach

**Teoria.** Im więcej akordów w wektorze, tym lepiej wektor „pamięta” akordy – więc miara „po staremu” (akordy zapytania w wektorze) rośnie niemal liniowo z `α`, aż do ok. 0,9. To nie poprawa jakości, tylko coraz silniejsze przepisywanie odpowiedzi. W zimnym starcie krzywa jest płaska dla A (zapytanie nie ma akordów, więc jego pozycja zależy tylko od nut), a dla B\* ma łagodne maksimum. Bracia (miara niezależna od akordów) **maleją**, gdy akordy dominują – akordy są grube (84 możliwe) i zlewają perfumy z różnych linii.

**Wniosek:** `alpha = 0,4` i `gain = 1,5` to decyzja projektowa (kompromis), a nie pomiar „prawdy”. Ostateczną miarą będą Twoje pary podobieństwa z `pairs_test.csv`.

In [13]:
def wiersz(model, wartosc, fn):
    wyciek = eval_notes(fn(X_note, X_acc))                            # miara "po staremu" (akordy zapytania w wektorze)
    zimny = eval_cold(fn, per_fold=1000)                              # uczciwie
    return {"model": model, "wartość": wartosc, "zgodność (wyciek)": wyciek.mean(), "zgodność (zimny start)": zimny.mean(),
            **{k: v for k, v in diag_full(fn(X_note, X_acc)).items() if k == "bracia"}}

wiersze = [wiersz("A: alpha", a, lambda Xn, Xa, a=a: build_A(Xn, Xa, alpha=a)) for a in [0.0, 0.4, 0.8]]
wiersze += [wiersz("B*: gain", g, lambda Xn, Xa, g=g: build_B(Xn, Xa, gain=g)[0]) for g in [0.0, 1.5, 2.5]]
display(pd.DataFrame(wiersze).round(4))

      model  wartość  zgodność (wyciek)  zgodność (zimny start)  bracia
0  A: alpha      0.0             0.5241                  0.5138  0.1553
1  A: alpha      0.4             0.8189                  0.5161  0.1522
2  A: alpha      0.8             0.9029                  0.5115  0.1009
3  B*: gain      0.0             0.6929                  0.6274  0.1256
4  B*: gain      1.5             0.8084                  0.6346  0.1120
5  B*: gain      2.5             0.8480                  0.6288  0.1019


**Wniosek:** miara „po staremu” rośnie z udziałem akordów niemal liniowo (A: 0,524 → 0,819 → 0,903; B\*: 0,693 → 0,808 → 0,848), a zimny start jest płaski dla A (0,512–0,516) i ma maksimum dla B\* przy `gain = 1,5` (0,6346 vs 0,6274 i 0,6288). Bracia **maleją**, gdy akordy dominują (A: 0,155 → 0,101; B\*: 0,126 → 0,102). Czyli `alpha` i `gain` to decyzje projektowe: zostają 0,4 i 1,5.


### 6.7 Czułość na `MIN_DF_NOTE` i na `DIM` w metodzie B

**Teoria.** Wyższy `MIN_DF_NOTE` usuwa rzadkie nuty. Dla A (z IDF) rzadkie nuty i tak niewiele dokładają; dla B mają mało współwystąpień, więc ich wektory są szumowe – ale zbyt wysoki próg wycina użyteczną informację. `DIM` wyznacza, ile osi wiernie odwzorowuje macierz PPMI.

Modele **tylko z nut** (uczciwa miara), różnice parowane względem ustawienia z `CONFIG`.

In [14]:
def B_nuty(Xn, dim=None):
    return build_B(Xn, None, dim=dim, use_accords=False, top1_weight=0.0, ppmi_shift=0.0)[0]

print("MIN_DF_NOTE (modele tylko z nut), różnica vs 3:")
wyn = {}
for md_ in [1, 3, 10, 50]:
    Xn_md, vocab_md, _ = to_sparse(note_features(), md_)
    wyn[md_] = (eval_notes(build_A(Xn_md, None)), eval_notes(B_nuty(Xn_md)), len(vocab_md),
                int((np.asarray(Xn_md.sum(axis=1)).ravel() == 0).sum()))
for md_, (a, b, nv, nz) in wyn.items():
    print(f"  MIN_DF={md_:3d} słownik {nv:5d} perfum bez nut {nz}   A {roznica(a, wyn[3][0])}   B {roznica(b, wyn[3][1])}")

print("\nDIM dla B (tylko nuty), różnica vs 256:")
wB = {d: eval_notes(B_nuty(X_note, d)) for d in [64, 128, 256, 512]}
for d, v in wB.items():
    print(f"  DIM={d:3d} zgodność {pm(v)}   różnica vs 256: {roznica(v, wB[256])}")

MIN_DF_NOTE (modele tylko z nut), różnica vs 3:
  MIN_DF=  1 słownik  1664 perfum bez nut 0   A -0.0002 ± 0.0002   B +0.0021 ± 0.0005
  MIN_DF=  3 słownik  1133 perfum bez nut 0   A +0.0000 ± 0.0000   B +0.0000 ± 0.0000
  MIN_DF= 10 słownik   745 perfum bez nut 0   A -0.0002 ± 0.0002   B -0.0009 ± 0.0006
  MIN_DF= 50 słownik   365 perfum bez nut 0   A -0.0006 ± 0.0002   B -0.0331 ± 0.0011

DIM dla B (tylko nuty), różnica vs 256:
  DIM= 64 zgodność 0.5280 ± 0.0018   różnica vs 256: -0.0260 ± 0.0009
  DIM=128 zgodność 0.5455 ± 0.0018   różnica vs 256: -0.0085 ± 0.0007
  DIM=256 zgodność 0.5541 ± 0.0018   różnica vs 256: +0.0000 ± 0.0000
  DIM=512 zgodność 0.5548 ± 0.0018   różnica vs 256: +0.0008 ± 0.0007


**Wniosek:** dla A `MIN_DF_NOTE` nie ma znaczenia (|Δ| ≤ 0,0006). Dla B: próg 1 daje niewielki zysk (+0,0021 ± 0,0005), 10 jest bez różnicy (−0,0009 ± 0,0006), 50 wyraźnie szkodzi (−0,033). Zostaje 3 (margines bezpieczeństwa przed perfumami bez nut). `DIM` dla B: 64 (−0,026) i 128 (−0,0085) gorsze, 512 bez poprawy (+0,0008 ± 0,0007) – zostaje 256.


## 7. Hybryda A + B\*: wybór udziału A (`HYBRID_W_A`)

**Teoria.** Po sklejeniu `[√w·A, √(1−w)·B*]` podobieństwo to `w·cos_A + (1−w)·cos_B*`. Zmieniając `w` przesuwamy się od czystego B\* (`w=0`) do czystego A (`w=1`). **Jednej „najlepszej” wartości nie ma**, bo trzy miary prowadzą w różne strony:
* zgodność akordów w zimnym starcie jest najwyższa blisko `w = 0`,
* bracia w top-10 rosną z `w`,
* hubness jest najniższy w okolicy `w ≈ 0,3`.

Wybieramy `w` z kompromisu: najmniejszy koszt zgodności przy wyraźnym zysku w pozostałych dwóch miarach. Zastrzeżenie: `shift`, wagę tokena i `w` dobrano na tym samym zbiorze i tej samej mierze, więc rzeczywista przewaga może być nieco mniejsza niż w tabeli.

In [15]:
Z_Bs_pel = build_B(X_note, X_acc)[0]
Z_A_pel = build_A(X_note, X_acc)
wiersze, zimne_h = [], {}
for w_A in [0.0, 0.15, 0.30, 0.50, 1.0]:
    if w_A == 0:
        fn, Z = (lambda Xn, Xa: build_B(Xn, Xa)[0]), Z_Bs_pel
    elif w_A == 1:
        fn, Z = (lambda Xn, Xa: build_A(Xn, Xa)), Z_A_pel
    else:
        fn = lambda Xn, Xa, w=w_A: hybrid(build_A(Xn, Xa), build_B(Xn, Xa)[0], w)
        Z = hybrid(Z_A_pel, Z_Bs_pel, w_A)
    zimne_h[w_A] = eval_cold(fn)
    wiersze.append({"w_A": w_A, "zgodność (zimny start)": zimne_h[w_A].mean(), "SE": zimne_h[w_A].std(ddof=1) / np.sqrt(N), **diag_full(Z)})
tab_h = pd.DataFrame(wiersze); display(tab_h.round(4))
print("różnica zgodności względem czystego B* (w_A = 0):")
for w_A in [0.15, 0.30, 0.50, 1.0]:
    print(f"  w_A = {w_A:4.2f}: {roznica(zimne_h[w_A], zimne_h[0.0])}")

    w_A  zgodność (zimny start)      SE  bracia  hubness  anty_huby
0  0.00                  0.6336  0.0011  0.1120     2.48      0.089
1  0.15                  0.6333  0.0010  0.1255     2.30      0.074
2  0.30                  0.6287  0.0010  0.1343     2.24      0.059
3  0.50                  0.6157  0.0010  0.1442     2.27      0.046
4  1.00                  0.5175  0.0009  0.1522     2.59      0.027
różnica zgodności względem czystego B* (w_A = 0):
  w_A = 0.15: -0.0002 ± 0.0002
  w_A = 0.30: -0.0048 ± 0.0003
  w_A = 0.50: -0.0178 ± 0.0004
  w_A = 1.00: -0.1161 ± 0.0008


**Wniosek:** udział A do 0,15 nic nie kosztuje (−0,0002 ± 0,0002, w granicach szumu), a daje więcej braci (0,112 → 0,126) i niższy hubness (2,48 → 2,30). Przy 0,30 zgodność spada o 0,0048 ± 0,0003, ale bracia rosną do 0,134, hubness do 2,24 i anty-huby do 5,9%. Przy 0,50 koszt rośnie do 0,018. **`HYBRID_W_A = 0,30`** to wybór z preferencją dla braci i hubness; jeśli ważniejsza jest sama zgodność akordów, ustaw 0,15 (bezpieczniejszy wariant) albo 0 (czyste B\*).



## 8. Podsumowanie: decyzje przeniesione do `embeddings_v3_proste`

| Parametr (`CONFIG`) | Wartość | Podstawa |
|---|---|---|
| `LAYER_WEIGHTS` | 1/1/1 | ustalone wcześniej; „baza ważniejsza” −0,050, „głowa ważniejsza” −0,0035 (6.1) |
| `LENGTH_NORM_EXP` | 0 | `len_exp = 1` pogarsza o ok. 0,04 (6.1) |
| `IDF_POWER` | 0,5 | brak IDF −0,015, pełne −0,022 (6.1) |
| `MIN_DF_NOTE` | 3 | dla A bez wpływu; B: 50 → −0,033, 1 → +0,002 (6.7) |
| `DIM` | 256 | 128 pogarsza (A −0,015, B −0,0085), 384/512 dla B bez poprawy (6.2, 6.7) |
| `A_ALPHA_ACCORD` | 0,4 | nie wyznaczalne miarą opartą na akordach; zimny start płaski (6.6) |
| `B_CTX_SMOOTH` | 0,5 | 0,25: −0,036; 0,75: −0,010 (6.5) |
| `B_ACCORD_GAIN` | 1,5 | maksimum zimnego startu; bracia maleją przy większych wartościach (6.6) |
| `B_TOP1_WEIGHT` | 0,75 | +0,0276 (6.4) |
| `B_PPMI_SHIFT` | 0,69 | +0,0105; razem z tokenem +0,0504 (6.4) |
| `HYBRID_W_A` | 0,30 | kompromis zgodność / bracia / hubness (7); 0,15 to bezpieczniejsza alternatywa |
| usunięcie wspólnego kierunku w B | tak | bez tego −0,015 (6.5) |
| `eig` w B | 0 | +0,005 możliwe, ale niewłączone (6.5) |

**Czego tu celowo nie ma:**
* **Scalania nut szczegółowych w ogólne** – zrobisz je przy budowie bazy. Wiedz, że po scaleniu wektory trzeba policzyć ponownie; pomiary poza tym notatnikiem pokazywały zysk rzędu +0,01 zgodności (część tej różnicy to efekt „zagęszczenia” cech, nie nowa informacja).
* **Korekt hubness** (centrowanie, usuwanie głównych składowych, CSLS) – pomiary poza tym notatnikiem pokazały, że obniżają zgodność akordów, a hybryda z A obniża hubness skuteczniej.
* **Ostatecznego sprawdzianu** – wszystkie miary są pośrednie (akordy to głosy użytkowników). Parametry dobrano na tym samym zbiorze i tej samej mierze, więc rzeczywista przewaga może być mniejsza niż w tabelach. Ostateczną miarą powinny być Twoje pary podobieństwa z `pairs_test.csv` (np. odsetek par, w których podobna para ma wyższy cosinus niż niepodobna).

**Następny krok:** uruchom `embeddings_v3_proste` – tam powstają wektory końcowe (`Z_final.npy`), ocena czterech modeli, kontrola na znanych perfumach i zapis plików.
